# Predicting Dengue Cases Using Climate and Environmental Data

This notebook summarises a data science project investigating whether climate and environmental variables can improve **one-week-ahead dengue case forecasting** for Maynas, Loreto, Peru.

The project combines public dengue surveillance data with environmental predictors such as rainfall, temperature, humidity, wind speed, pressure and vegetation indicators. The final aim is to compare simple baselines, machine-learning models and traditional time-series methods using a chronological evaluation design.

## Project Aim

The aim of this project is to predict weekly dengue case counts one week ahead and assess whether climate and environmental predictors improve forecasting performance beyond recent dengue history and seasonality.

The target variable is weekly `dengue_cases`. For each target week `t`, the model only uses information available up to week `t-1`, helping avoid target-week data leakage.

## Data Sources

The project uses publicly available disease surveillance, climate and environmental data.

| Source | Data Used |
|---|---|
| OpenDengue | Historical dengue surveillance case counts |
| ERA5 / CFSR reanalysis | Temperature, humidity, wind speed, pressure and atmospheric variables |
| NOAA PERSIANN | Precipitation data |
| NOAA NDVI / MODIS NDVI | Vegetation index data |
| NOAA GHCN-Daily | Ground-based weather observations used during exploration |

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## Load The Modelling Dataset

The project uses an engineered one-week-ahead modelling dataset. The path below assumes the notebook is being run from the project repository root.

In [ ]:
DATA_PATH = "data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["week_start_date"])
df.head()

## Data Overview

This section checks the shape, date coverage and main target field before modelling.

In [ ]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Date range:", df["week_start_date"].min(), "to", df["week_start_date"].max())

if "dengue_cases" in df.columns:
    display(df[["week_start_date", "dengue_cases"]].head())

In [ ]:
df.info()

## Exploratory Data Analysis

The dengue case distribution is highly right-skewed, meaning most weeks have relatively low case counts but a smaller number of epidemic weeks have much higher values. This makes it important to report both MAE and RMSE during model evaluation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["dengue_cases"].dropna(), bins=40, ax=axes[0])
axes[0].set_title("Distribution of weekly dengue cases")
axes[0].set_xlabel("Weekly dengue cases")

sns.lineplot(data=df, x="week_start_date", y="dengue_cases", ax=axes[1])
axes[1].set_title("Weekly dengue cases over time")
axes[1].set_xlabel("Week")
axes[1].set_ylabel("Dengue cases")

plt.tight_layout()
plt.show()

## Chronological Train, Validation And Test Split

The modelling workflow uses a fixed chronological split:

| Period | Years | Purpose |
|---|---:|---|
| Training | 2000-2017 | Fit candidate models |
| Validation | 2018-2020 | Model comparison and selection |
| Final test | 2021-2023 | Final untouched holdout evaluation |

The final test period is only used after model choices are frozen.

In [ ]:
train = df[df["week_start_date"].dt.year <= 2017]
validation = df[(df["week_start_date"].dt.year >= 2018) & (df["week_start_date"].dt.year <= 2020)]
test = df[df["week_start_date"].dt.year >= 2021]

split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Final test"],
    "rows": [len(train), len(validation), len(test)],
    "start_date": [train["week_start_date"].min(), validation["week_start_date"].min(), test["week_start_date"].min()],
    "end_date": [train["week_start_date"].max(), validation["week_start_date"].max(), test["week_start_date"].max()],
})

split_summary

## Models Compared

The project compared simple benchmarks, machine-learning models and traditional time-series models.

| Model Type | Models |
|---|---|
| Baseline | Persistence, Ridge regression |
| Machine learning | Random Forest, XGBoost |
| Time series | SARIMA, SARIMAX |

Persistence predicts the next week using the previous week's dengue cases. This is a strong benchmark because dengue incidence is highly autocorrelated at a weekly horizon.

## Final Model Comparison

The table below summarises the final frozen evaluation on the 2021-2023 holdout period.

In [ ]:
final_results = pd.DataFrame({
    "Model": ["XGBoost", "Persistence", "XGBoost", "SARIMA"],
    "Feature set": ["History + climate", "Previous week's cases", "History only", "History + seasonality"],
    "MAE": [11.13, 11.22, 11.45, 12.14],
    "RMSE": [21.22, 22.00, 22.35, 21.67],
    "R2": [0.855, 0.844, 0.839, 0.849]
})

final_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_data = final_results.copy()
plot_data["Label"] = plot_data["Model"] + "\n" + plot_data["Feature set"]

sns.barplot(data=plot_data, x="Label", y="MAE", ax=axes[0], color="#3b82f6")
axes[0].set_title("Final test MAE by model")
axes[0].set_xlabel("")
axes[0].set_ylabel("MAE")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=plot_data, x="Label", y="RMSE", ax=axes[1], color="#14b8a6")
axes[1].set_title("Final test RMSE by model")
axes[1].set_xlabel("")
axes[1].set_ylabel("RMSE")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## Interpreting The Final Results

The best final model was **XGBoost with dengue history and climate/environmental predictors**, with:

- MAE: **11.13**
- RMSE: **21.22**
- R²: **0.855**

The improvement over persistence was modest, but the climate-enhanced XGBoost model achieved the lowest MAE, lowest RMSE and highest R² across the final model set.

## Feature Importance Summary

The final XGBoost model was still dominated by recent dengue incidence. The most important predictor was the one-week dengue lag, meaning recent case burden carried most of the short-term forecasting signal.

Important non-dengue features included seasonality, lagged precipitation, lagged temperature, wind speed and surface pressure. These variables provided smaller refinements rather than replacing the predictive value of recent dengue history.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": [
        "dengue_cases_lag1",
        "annual seasonal sine term",
        "precipitation_lag4",
        "temperature_lag1",
        "wind_speed_lag12",
        "surface_pressure_lag8",
        "wind_speed_rolling4",
        "precipitation_rolling4"
    ],
    "Importance (%)": [62.6, 9.4, 6.9, 5.4, 3.5, 3.0, 2.4, 1.6]
})

plt.figure(figsize=(9, 5))
sns.barplot(data=feature_importance, y="Feature", x="Importance (%)", color="#2563eb")
plt.title("Selected final XGBoost feature importance")
plt.xlabel("Relative importance (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

## Conclusion

The project found that weekly dengue cases can be forecast reasonably well one week ahead, largely because recent dengue incidence is highly predictive. Climate and environmental variables added some value in the final XGBoost model, but the improvement was incremental rather than transformative.

The final conclusion is that climate-enhanced machine learning can improve short-term dengue forecasting, but recent dengue history remains the strongest predictor and simple baselines should always be used as a benchmark.